In [ ]:
from importlib import reload
from adaptive_trade_extensions import make_ret_grid
import adaptive_reward_checkpoint_fresh
reload(adaptive_reward_checkpoint_fresh)

from adaptive_reward_checkpoint_fresh import run_adaptive_reward_yfinance_scheduled_flat_start_loop

live_res = run_adaptive_reward_yfinance_scheduled_flat_start_loop(
    checkpoint_path="output_adaptive_reward_live_TQQQ_20260605/live_checkpoint.joblib",
    output_dir="output_adaptive_reward_live_TQQQ_20260608",
    save_snapshot_path="output_adaptive_reward_live_TQQQ_20260608/live_checkpoint.joblib",

    start_date="2026-06-08",
    market_open_time="09:30",
    market_close_time="16:00",
    timezone="America/New_York",
    threshold_ret_grid_override=make_ret_grid(-0.5, 2.5, 0.05),

    initial_capital=100000.0,
    fee_pct=0.0,
    poll_seconds=60,

    preopen_update=True,
    wait_until_start=True,
    verbose=True,
)

print("latest checkpoint:", live_res["latest_checkpoint_path"])
display(live_res["trades_df"].tail())
display(live_res["daily_log_df"].tail())

In [ ]:
from adaptive_reward_checkpoint_fresh import run_adaptive_reward_yfinance_day_lookback
lookback_res = run_adaptive_reward_yfinance_day_lookback(
    checkpoint_path="output_adaptive_reward_live_TQQQ_20260603/live_checkpoint.joblib",
    lookback_date="2026-06-04",
    output_dir="output_adaptive_reward_lookback",
    compare_live_output_dir="output_adaptive_reward_live_TQQQ_20260604",

    market_open_time="09:30",
    market_close_time="16:00",
    timezone="America/New_York",

    initial_capital=100000.0,
    fee_pct=0.0,

    starting_position_qty=600,
    starting_position_entry_px=86.57,
    # optional: starting_cash=47650.0,

    refresh_data=True,
    verbose=True,
)

print(lookback_res["lookback_output_dir"])
print(lookback_res["comparison"])
display(lookback_res["signal_decisions_all_df"].tail())
display(lookback_res["trades_df"].tail())
display(lookback_res["daily_log_df"].tail())

In [2]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot_chronological,
)

from adaptive_trade_extensions import (
    RollingThresholdConfig,
    make_threshold_grid,
    make_ret_grid,
)

five_min_grid = make_ret_grid(0, 25, 0.5)

common_kwargs = dict(
    daily_csv_path="DataAPI/data/TQQQ_day.csv",
    k5m_csv_path="DataAPI/data/TQQQ_5M.csv",
    code="TQQQ",
    daily_chan_start="2010-02-11",
    accumulation_start="2012-02-11",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=252,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=five_min_grid,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={"vix_": "VIX.csv"},
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)

build_dir = Path("output_TQQQ_build_prev_daily_context_at_2020")
checkpoint_path = build_dir / "fresh_checkpoint_prev_daily_context__before_2020.joblib"

# Build the clean pre-2020 checkpoint only if it does not already exist.
if checkpoint_path.exists():
    print("Using existing clean checkpoint:", checkpoint_path)
else:
    build_res = build_adaptive_reward_snapshot(
        snapshot_path=str(checkpoint_path),
        snapshot_end_time="2019-12-31 16:00:00",
        output_dir=str(build_dir),
        autosave_year_start_checkpoints=False,
        **common_kwargs,
    )

    print("Built checkpoint:", build_res["snapshot_path"])
    display(build_res["daily_reward_df"].tail())
    display(build_res["daily_log_df"].tail())

base_output_dir = Path("output_TQQQ_chrono_prev_daily_context_wide_5m_resumable")
base_output_dir.mkdir(parents=True, exist_ok=True)

chunks = [
    ("2020", "2020-12-31 16:00:00"),
    ("2021", "2021-12-31 16:00:00"),
    ("2022", "2022-12-31 16:00:00"),
    ("2023", "2023-12-31 16:00:00"),
    ("2024", "2024-12-31 16:00:00"),
    ("2025", "2025-12-31 16:00:00"),
    ("2026", "2026-06-12 16:00:00"),
]

current_checkpoint = checkpoint_path
last_res = None

for year, chunk_end_time in chunks:
    chunk_dir = base_output_dir / f"chunk_{year}"
    chunk_checkpoint = chunk_dir / f"continued_checkpoint__through_{year}.joblib"

    # If this chunk already finished in a previous run, skip it and resume from it.
    if chunk_checkpoint.exists():
        print(f"[SKIP] {year} already completed:", chunk_checkpoint)
        current_checkpoint = chunk_checkpoint
        continue

    print(f"[RUN] {year}: {current_checkpoint} -> {chunk_end_time}")

    is_first_chunk = Path(current_checkpoint) == checkpoint_path

    last_res = run_adaptive_reward_from_snapshot_chronological(
        snapshot_path=str(current_checkpoint),
        end_time=chunk_end_time,

        # Only needed for the first chunk from the clean pre-2020 checkpoint.
        sim_start="2020-01-01 09:30:00" if is_first_chunk else None,
        trade_start="2020-01-01 09:30:00" if is_first_chunk else None,

        output_dir=str(chunk_dir),
        save_snapshot_path=str(chunk_checkpoint),

        # First chunk starts a fresh portfolio. Later chunks preserve state from checkpoint.
        reset_execution_state=True if is_first_chunk else False,
        initial_capital=common_kwargs["initial_capital"],
        fee_pct=common_kwargs["fee_pct"],

        dp_lookback_override=common_kwargs["dp_lookback"],
        daily_threshold_lookback_days_override=252,
        threshold_ret_grid_override=five_min_grid,

        autosave_year_start_checkpoints=True,
        verbose=True,
    )

    print(f"[DONE] {year} checkpoint:", chunk_checkpoint)
    current_checkpoint = chunk_checkpoint

print("Final checkpoint:", current_checkpoint)

[TRAIN][DAILY-PROB] n=200 pos=49 (24.50%)
[TRAIN][DAILY-PROB] n=225 pos=52 (23.11%)
[TRAIN][DAILY-PROB] n=250 pos=61 (24.40%)
[TRAIN][DAILY-PROB] n=275 pos=65 (23.64%)
[TRAIN][DAILY-PROB] n=300 pos=76 (25.33%)
[TRAIN][DAILY-PROB] n=325 pos=76 (23.38%)
[TRAIN][DAILY-PROB] n=350 pos=76 (21.71%)
[TRAIN][DAILY-PROB] n=375 pos=79 (21.07%)
[TRAIN][DAILY-PROB] n=400 pos=84 (21.00%)
[TRAIN][DAILY-PROB] n=425 pos=91 (21.41%)
[TRAIN][DAILY-PROB] n=450 pos=96 (21.33%)
[TRAIN][DAILY-PROB] n=475 pos=99 (20.84%)
[TRAIN][DAILY-PROB] n=500 pos=104 (20.80%)
[TRAIN][DAILY-PROB] n=525 pos=114 (21.71%)
[TRAIN][DAILY-PROB] n=550 pos=118 (21.45%)
[TRAIN][DAILY-PROB] n=575 pos=120 (20.87%)
[TRAIN][DAILY-PROB] n=600 pos=128 (21.33%)
[TRAIN][DAILY-PROB] n=625 pos=137 (21.92%)
[TRAIN][DAILY-PROB] n=650 pos=142 (21.85%)
[TRAIN][DAILY-PROB] n=675 pos=150 (22.22%)
[TRAIN][DAILY-PROB] n=700 pos=152 (21.71%)
[TRAIN][DAILY-PROB] n=725 pos=155 (21.38%)
[TRAIN][DAILY-PROB] n=750 pos=160 (21.33%)
[TRAIN][DAILY-PROB] n=7

,date,decision_date,result_date,p_day,p_day_source_date,daily_gate_mode,daily_reward_mode,direct_gate_confidence,direct_gate_prob_force_buy,direct_gate_prob_free,...,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
1979,2019-12-23,2019-12-23,2019-12-24,0.155877,2019-12-23,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,0.000121,0.000121,0.000121,0.000121,FORCE_BUY,9.917156e+09,10.7325,2.0,5.5
1980,2019-12-24,2019-12-24,2019-12-26,0.146876,2019-12-24,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,0.029743,0.006162,0.001980,0.029743,FORCE_BUY,1.021213e+10,11.0787,0.0,0.0
1981,2019-12-26,2019-12-26,2019-12-27,0.159621,2019-12-26,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,-0.014070,0.000797,-0.000117,-0.014070,FREE,1.022026e+10,10.9525,0.0,0.0
1982,2019-12-27,2019-12-27,2019-12-30,0.206062,2019-12-27,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,-0.021668,-0.023506,0.000118,-0.021668,FORCE_SELL,1.022147e+10,10.7775,0.0,0.0
1983,2019-12-30,2019-12-30,2019-12-31,0.228122,2019-12-30,threshold,counterfactual_5m,NaN,NaN,NaN,...,FREE,0.007886,0.003475,0.003475,0.003475,FORCE_BUY,1.030207e+10,10.8763,4.0,3.0


,date,equity,cash,pos,qty,entry_px,decision_date,result_date,buy_th,sell_th,...,daily_buy_level,daily_sell_level,decision_extreme_base_dir,decision_extreme_region,decision_extreme_label,decision_extreme_window_end_date,decision_extreme_ref_high,decision_extreme_ref_low,decision_extreme_future_max_high,decision_extreme_future_min_low
1979,2019-12-24,171582.896770,0.000000,1,15987.225415,9.585,2019-12-23,2019-12-24,0.0,0.0,...,0.285,0.305,sell,sell_high_broken,0.0,2019-12-31,10.7725,10.6962,11.1325,10.6
1980,2019-12-26,177117.674208,0.000000,1,15987.225415,9.585,2019-12-24,2019-12-26,0.0,0.0,...,0.285,0.305,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1981,2019-12-27,175100.086361,0.000000,1,15987.225415,9.585,2019-12-26,2019-12-27,0.0,0.0,...,0.285,0.305,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1982,2019-12-30,172302.321913,0.000000,1,15987.225415,9.585,2019-12-27,2019-12-30,4.0,3.0,...,0.220,0.265,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1983,2019-12-31,173120.867855,173120.867855,0,0.000000,NaN,2019-12-30,2019-12-31,4.0,4.0,...,0.220,0.265,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN


[RUN] 2020: output_TQQQ_build_prev_daily_context_at_2020\fresh_checkpoint_prev_daily_context__before_2020.joblib -> 2020-12-31 16:00:00
[CHRONO] 5m day=2019-12-31 rows=253260:253294
[TRAIN][5M] asof=2019-12-31 feats=114 buy=YES sell=YES rows=57752
[CHECKPOINT] autosaved chronological year-start snapshot for 2020 before 2020-01-02 -> output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2020\year_start_checkpoints\continued_checkpoint__through_2020__start_2020.joblib
[CHRONO] 5m day=2020-01-02 rows=253294:253473
[CHRONO] daily close=2020-01-02 idx=2489
[CHRONO] 5m day=2020-01-03 rows=253473:253657
[CHRONO] daily close=2020-01-03 idx=2490
[CHRONO] 5m day=2020-01-06 rows=253657:253836
[TRAIN][5M] asof=2020-01-06 feats=114 buy=YES sell=YES rows=57885
[CHRONO] daily close=2020-01-06 idx=2491
[CHRONO] 5m day=2020-01-07 rows=253836:254014
[CHRONO] daily close=2020-01-07 idx=2492
[CHRONO] 5m day=2020-01-08 rows=254014:254203
[CHRONO] daily close=2020-01-08 idx=2493
[CHRONO] 5m day=2020

c:\Users\TonyTang\Documents\chan.py\adaptive_reward_checkpoint_fresh.py:3132: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  daily_log_df = pd.concat(daily_log_frames, ignore_index=True, sort=False) if daily_log_frames else pd.DataFrame()


...

[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\trades.csv
[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\signal_decisions.csv
[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\daily_log.csv
[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\daily_reward_log.csv
[CHECKPOINT] saved chronological snapshot -> output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\continued_checkpoint__through_2024.joblib
[DONE] 2024 checkpoint: output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\continued_checkpoint__through_2024.joblib
[RUN] 2025: output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\continued_checkpoint__through_2024.joblib -> 2025-12-31 16:00:00
[CHRONO] 5m day=2024-12-31 rows=491974:492021
[TRAIN][5M] asof=2024-12-31 feats=114 buy=YES sell=YES rows=107549
[CHECKPOINT] autosaved chronological year-start snapshot for 2025 before 2025-01-02 -> ou

c:\Users\TonyTang\Documents\chan.py\adaptive_reward_checkpoint_fresh.py:3132: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  daily_log_df = pd.concat(daily_log_frames, ignore_index=True, sort=False) if daily_log_frames else pd.DataFrame()


[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\trades.csv
[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\signal_decisions.csv
[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\daily_log.csv
[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\daily_reward_log.csv
[CHECKPOINT] saved chronological snapshot -> output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\continued_checkpoint__through_2025.joblib
[DONE] 2025 checkpoint: output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\continued_checkpoint__through_2025.joblib
[RUN] 2026: output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\continued_checkpoint__through_2025.joblib -> 2026-06-12 16:00:00
[CHRONO] 5m day=2025-12-31 rows=539863:539910
[TRAIN][5M] asof=2025-12-31 feats=114 buy=YES sell=YES rows=117379
[CHECKPOINT] autosaved chronological year-start snapshot for 2026 before 2026-01-02 -> ou

c:\Users\TonyTang\Documents\chan.py\adaptive_reward_checkpoint_fresh.py:3132: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  daily_log_df = pd.concat(daily_log_frames, ignore_index=True, sort=False) if daily_log_frames else pd.DataFrame()


[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\trades.csv
[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\signal_decisions.csv
[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\daily_log.csv
[SAVED] output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\daily_reward_log.csv
[CHECKPOINT] saved chronological snapshot -> output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\continued_checkpoint__through_2026.joblib
[DONE] 2026 checkpoint: output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\continued_checkpoint__through_2026.joblib
Final checkpoint: output_TQQQ_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\continued_checkpoint__through_2026.joblib


In [ ]:
from adaptive_reward_checkpoint_fresh import run_adaptive_reward_from_snapshot

checkpoint_path = r"output_adaptive_reward_snapshot_build_TQQQ_252days_at_2020\year_start_checkpoints\final_checkpoint__start_2026.joblib"

res = run_adaptive_reward_from_snapshot(
    snapshot_path=checkpoint_path,

    # Simulate through this local-data timestamp.
    end_time="2026-06-12 16:00:00",

    # Actual trading begins here.
    # The first trading day will use the latest prior p_day/gate from the checkpoint/history.
    trade_start="2026-01-01 09:30:00",

    # Optional; defaults to checkpoint_time + 1 day.
    # Keep this at or before trade_start if you want all trade_start signals considered.
    sim_start="2026-01-01 09:30:00",

    output_dir=r"output_prev_checkpoint_local_TQQQ_20260101_20260612_W_regime",
    save_snapshot_path=r"output_prev_checkpoint_local_TQQQ_20260101_20260612\continued_checkpoint.joblib",

    # Start clean/flat from this capital.
    reset_execution_state=True,
    initial_capital=100000.0,
    fee_pct=0.0,

    # Keep saved model/checkpoint logic.
    autosave_year_start_checkpoints=False,
    verbose=True,
)

print("Output dir:", res["output_dir"])
print("Continued checkpoint:", res["continued_snapshot_path"])
print(res["daily_reward_df"].tail())
print(res["signal_decisions_all_df"].tail())

In [ ]:

from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "TQQQ_adaptive_reward_fresh_start_252days_at_2020.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/TQQQ_day.csv",
    k5m_csv_path="DataAPI/data/TQQQ_5M.csv",
    code="TQQQ",
    daily_chan_start="2010-02-11",
    accumulation_start="2012-02-11",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)

snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2019-12-31",
    output_dir="output_adaptive_reward_snapshot_build_TQQQ_252days_at_2020",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())

resume_res = run_adaptive_reward_from_snapshot(
    dp_lookback_override=5,
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2020-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/TQQQ_adaptive_reward_fresh_start__continued.joblib
    #save_snapshot_path="output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020/final_checkpoint.joblib",
    autosave_year_start_checkpoints=True,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())



In [2]:

from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "TQQQ_adaptive_reward_fresh_start_252days_at_2020.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/TQQQ_day.csv",
    k5m_csv_path="DataAPI/data/TQQQ_5M.csv",
    code="TQQQ",
    daily_chan_start="2012-01-01",
    accumulation_start="2014-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)

snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2019-12-31",
    output_dir="output_adaptive_reward_snapshot_build_TQQQ_252days_at_2020",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())

resume_res = run_adaptive_reward_from_snapshot(
    #dp_lookback_override=5,
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2020-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/TQQQ_adaptive_reward_fresh_start__continued.joblib
    #save_snapshot_path="output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020/final_checkpoint.joblib",
    autosave_year_start_checkpoints=True,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())



[TRAIN][DAILY-PROB] n=200 pos=50 (25.00%)
[TRAIN][DAILY-PROB] n=225 pos=53 (23.56%)
[TRAIN][DAILY-PROB] n=250 pos=57 (22.80%)
[TRAIN][DAILY-PROB] n=275 pos=62 (22.55%)
[TRAIN][DAILY-PROB] n=300 pos=63 (21.00%)
[TRAIN][DAILY-PROB] n=325 pos=66 (20.31%)
[TRAIN][DAILY-PROB] n=350 pos=71 (20.29%)
[TRAIN][DAILY-PROB] n=375 pos=72 (19.20%)
[TRAIN][DAILY-PROB] n=400 pos=82 (20.50%)
[TRAIN][DAILY-PROB] n=425 pos=90 (21.18%)
[TRAIN][DAILY-PROB] n=450 pos=90 (20.00%)
[TRAIN][DAILY-PROB] n=475 pos=91 (19.16%)
[TRAIN][DAILY-PROB] n=500 pos=98 (19.60%)
[TRAIN][DAILY-PROB] n=525 pos=105 (20.00%)
[TRAIN][DAILY-PROB] n=550 pos=110 (20.00%)
[TRAIN][DAILY-PROB] n=575 pos=116 (20.17%)
[TRAIN][DAILY-PROB] n=600 pos=122 (20.33%)
[TRAIN][DAILY-PROB] n=625 pos=124 (19.84%)
[TRAIN][DAILY-PROB] n=650 pos=131 (20.15%)
[TRAIN][DAILY-PROB] n=675 pos=137 (20.30%)
[TRAIN][DAILY-PROB] n=700 pos=141 (20.14%)
[TRAIN][DAILY-PROB] n=725 pos=154 (21.24%)
[TRAIN][DAILY-PROB] n=750 pos=159 (21.20%)
[TRAIN][DAILY-PROB] n=77

,date,decision_date,result_date,p_day,p_day_source_date,daily_gate_mode,daily_reward_mode,direct_gate_confidence,direct_gate_prob_force_buy,direct_gate_prob_free,...,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
1504,2019-12-20,2019-12-20,2019-12-23,0.191901,2019-12-20,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,0.002572,0.004443,0.003161,0.002572,FREE,2.733276e+11,10.7212,-0.5,-0.5
1505,2019-12-23,2019-12-23,2019-12-24,0.198899,2019-12-23,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,0.000121,-0.000571,0.002106,0.000121,FORCE_SELL,2.739033e+11,10.7325,-0.5,-0.5
1506,2019-12-24,2019-12-24,2019-12-26,0.169106,2019-12-24,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,0.029743,0.006162,0.001980,0.029743,FORCE_BUY,2.820501e+11,11.0787,-0.5,-0.5
1507,2019-12-26,2019-12-26,2019-12-27,0.195538,2019-12-26,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,-0.014070,0.000797,-0.000117,-0.014070,FREE,2.822748e+11,10.9525,-0.5,-0.5
1508,2019-12-27,2019-12-27,2019-12-30,0.265744,2019-12-27,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_SELL,-0.021668,-0.023506,0.000118,0.000118,FORCE_SELL,2.823081e+11,10.7775,-0.5,-0.5


,date,equity,cash,pos,qty,entry_px,decision_date,result_date,buy_th,sell_th,...,daily_buy_level,daily_sell_level,decision_extreme_base_dir,decision_extreme_region,decision_extreme_label,decision_extreme_window_end_date,decision_extreme_ref_high,decision_extreme_ref_low,decision_extreme_future_max_high,decision_extreme_future_min_low
1504,2019-12-23,258710.022266,0.00000,1,24130.696402,10.51,2019-12-20,2019-12-23,-0.5,-0.5,...,0.200,0.250,sell,sell_high_broken,0.0,2019-12-30,10.6900,10.5750,11.1325,10.6
1505,2019-12-24,258982.699135,0.00000,1,24130.696402,10.51,2019-12-23,2019-12-24,-0.5,-0.5,...,0.315,0.335,sell,sell_high_broken,0.0,2019-12-31,10.7725,10.6962,11.1325,10.6
1506,2019-12-26,267336.746229,0.00000,1,24130.696402,10.51,2019-12-24,2019-12-26,-0.5,-0.5,...,0.315,0.335,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1507,2019-12-27,264291.452343,0.00000,1,24130.696402,10.51,2019-12-26,2019-12-27,-0.5,-0.5,...,0.200,0.250,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1508,2019-12-30,265859.947610,265859.94761,0,0.000000,NaN,2019-12-27,2019-12-30,-0.5,2.4,...,0.200,0.250,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN


[TRAIN][DAILY-PROB] n=1861 pos=372 (19.99%)
[TRAIN][DAILY-PROB] n=1886 pos=377 (19.99%)
[TRAIN][DAILY-PROB] n=1911 pos=379 (19.83%)
[TRAIN][DAILY-PROB] n=1936 pos=380 (19.63%)
[TRAIN][DAILY-PROB] n=1961 pos=382 (19.48%)
[TRAIN][DAILY-PROB] n=1986 pos=386 (19.44%)
[TRAIN][DAILY-PROB] n=2011 pos=389 (19.34%)
[TRAIN][DAILY-PROB] n=2036 pos=395 (19.40%)
[TRAIN][DAILY-PROB] n=2061 pos=406 (19.70%)
[TRAIN][DAILY-PROB] n=2086 pos=407 (19.51%)
[TRAIN][DAILY-PROB] n=2111 pos=410 (19.42%)
[TRAIN][DAILY-PROB] n=2136 pos=418 (19.57%)
[TRAIN][DAILY-PROB] n=2161 pos=421 (19.48%)
[TRAIN][DAILY-PROB] n=2186 pos=427 (19.53%)
[TRAIN][DAILY-PROB] n=2211 pos=427 (19.31%)
[TRAIN][DAILY-PROB] n=2236 pos=430 (19.23%)
[TRAIN][DAILY-PROB] n=2261 pos=433 (19.15%)
[TRAIN][DAILY-PROB] n=2286 pos=444 (19.42%)
[TRAIN][DAILY-PROB] n=2311 pos=447 (19.34%)
[TRAIN][DAILY-PROB] n=2336 pos=453 (19.39%)
[TRAIN][DAILY-PROB] n=2361 pos=459 (19.44%)
[TRAIN][DAILY-PROB] n=2386 pos=461 (19.32%)
[TRAIN][DAILY-PROB] n=2411 pos=4

,date,decision_date,result_date,p_day,p_day_source_date,daily_gate_mode,daily_reward_mode,direct_gate_confidence,direct_gate_prob_force_buy,direct_gate_prob_free,...,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
3125,2026-06-05,2026-06-05,2026-06-08,0.380011,2026-06-05,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_SELL,0.017305,0.011502,0.011502,0.011502,FORCE_BUY,1.143277e+23,76.0639,-0.5,-0.5
3126,2026-06-08,2026-06-08,2026-06-09,0.278875,2026-06-08,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,-0.060625,0.005896,0.000000,-0.060625,FREE,1.150018e+23,73.2900,-0.5,-0.5
3127,2026-06-09,2026-06-09,2026-06-10,0.370303,2026-06-09,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_SELL,-0.060559,-0.063389,-0.016276,-0.016276,FORCE_SELL,1.131301e+23,68.1095,-0.5,-0.5
3128,2026-06-10,2026-06-10,2026-06-11,0.303717,2026-06-10,threshold,counterfactual_5m,NaN,NaN,NaN,...,FREE,0.070863,0.067446,0.000000,0.067446,FORCE_BUY,1.211468e+23,76.9629,-0.5,-0.5
3129,2026-06-11,2026-06-11,2026-06-12,0.283013,2026-06-11,threshold,counterfactual_5m,NaN,NaN,NaN,...,FREE,0.029686,0.072628,0.000000,0.072628,FREE,1.299454e+23,78.0399,-0.5,-0.5


,date,equity,cash,pos,qty,entry_px,decision_date,result_date,buy_th,sell_th,...,daily_buy_level,daily_sell_level,decision_extreme_base_dir,decision_extreme_region,decision_extreme_label,decision_extreme_window_end_date,decision_extreme_ref_high,decision_extreme_ref_low,decision_extreme_future_max_high,decision_extreme_future_min_low
1616,2026-06-08,1.762186e+06,1.762186e+06,0,0.000000,NaN,2026-06-05,2026-06-08,-0.5,-0.5,...,0.305,0.345,sell,sell_high_held,1.0,2026-06-12,82.08,72.68,79.3407,66.79
1617,2026-06-09,1.655353e+06,0.000000e+00,1,22586.342032,78.02,2026-06-08,2026-06-09,-0.5,-0.5,...,0.305,0.345,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1618,2026-06-10,1.610858e+06,1.610858e+06,0,0.000000,NaN,2026-06-09,2026-06-10,-0.5,-0.5,...,0.275,0.325,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1619,2026-06-11,1.664750e+06,1.664750e+06,0,0.000000,NaN,2026-06-10,2026-06-11,-0.5,-0.5,...,0.275,0.325,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1620,2026-06-12,1.785658e+06,1.785658e+06,0,0.000000,NaN,2026-06-11,2026-06-12,-0.5,-0.5,...,0.275,0.325,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx,exec_idx,exec_ts
0,buy,203442,11.0550,9045.680687,0.0,ADAPTIVE_FORCE_BUY->first acceptable 5m signal,2020-01-02 06:30:00,1.123799,-0.5,FORCE_BUY,NaN,NaN,NaN,203443,2020-01-02 06:35:00
1,sell,203795,10.8012,9045.680687,0.0,5m SELL signal,2020-01-06 05:00:00,12.502633,-0.5,FREE,-2295.793758,11.0550,203442.0,203796,2020-01-06 05:05:00
2,buy,203803,10.8125,9036.227167,0.0,5m BUY signal,2020-01-06 05:45:00,1.196643,-0.5,FREE,NaN,NaN,NaN,203804,2020-01-06 05:55:00
3,sell,203813,10.8375,9036.227167,0.0,5m SELL signal,2020-01-06 07:05:00,11.537799,-0.5,FREE,225.905679,10.8125,203803.0,203814,2020-01-06 07:10:00
4,buy,203841,10.7812,9083.414826,0.0,5m BUY signal,2020-01-06 09:25:00,1.268677,-0.5,FREE,NaN,NaN,NaN,203842,2020-01-06 09:30:00


In [1]:

from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "TQQQ_adaptive_reward_fresh_start_252days_at_2020_W_Bonds.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/TQQQ_day.csv",
    k5m_csv_path="DataAPI/data/TQQQ_5M.csv",
    code="TQQQ",
    daily_chan_start="2012-01-01",
    accumulation_start="2018-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
        "US2Y": "US2Y.csv",
        #"US5Y": "US5Y.csv",
        "US10Y": "US10Y.csv",
        #"US30Y": "US30Y.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)

snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2019-12-31",
    output_dir="output_adaptive_reward_snapshot_build_TQQQ_252days_at_2020_W_Bonds",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())

resume_res = run_adaptive_reward_from_snapshot(
    #dp_lookback_override=5,
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2020-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020_W_Bonds",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/TQQQ_adaptive_reward_fresh_start__continued.joblib
    #save_snapshot_path="output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020_W_Bonds/final_checkpoint.joblib",
    autosave_year_start_checkpoints=True,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())



[TRAIN][DAILY-PROB] n=200 pos=50 (25.00%)
[TRAIN][DAILY-PROB] n=225 pos=53 (23.56%)
[TRAIN][DAILY-PROB] n=250 pos=57 (22.80%)
[TRAIN][DAILY-PROB] n=275 pos=62 (22.55%)
[TRAIN][DAILY-PROB] n=300 pos=63 (21.00%)
[TRAIN][DAILY-PROB] n=325 pos=66 (20.31%)
[TRAIN][DAILY-PROB] n=350 pos=71 (20.29%)
[TRAIN][DAILY-PROB] n=375 pos=72 (19.20%)
[TRAIN][DAILY-PROB] n=400 pos=82 (20.50%)
[TRAIN][DAILY-PROB] n=425 pos=90 (21.18%)
[TRAIN][DAILY-PROB] n=450 pos=90 (20.00%)
[TRAIN][DAILY-PROB] n=475 pos=91 (19.16%)
[TRAIN][DAILY-PROB] n=500 pos=98 (19.60%)
[TRAIN][DAILY-PROB] n=525 pos=105 (20.00%)
[TRAIN][DAILY-PROB] n=550 pos=110 (20.00%)
[TRAIN][DAILY-PROB] n=575 pos=116 (20.17%)
[TRAIN][DAILY-PROB] n=600 pos=122 (20.33%)
[TRAIN][DAILY-PROB] n=625 pos=124 (19.84%)
[TRAIN][DAILY-PROB] n=650 pos=131 (20.15%)
[TRAIN][DAILY-PROB] n=675 pos=137 (20.30%)
[TRAIN][DAILY-PROB] n=700 pos=141 (20.14%)
[TRAIN][DAILY-PROB] n=725 pos=154 (21.24%)
[TRAIN][DAILY-PROB] n=750 pos=159 (21.20%)
[TRAIN][DAILY-PROB] n=77

,date,decision_date,result_date,p_day,p_day_source_date,daily_gate_mode,daily_reward_mode,direct_gate_confidence,direct_gate_prob_force_buy,direct_gate_prob_free,...,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
497,2019-12-20,2019-12-20,2019-12-23,0.191901,2019-12-20,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,0.002572,0.004443,0.003161,0.002572,FREE,3.962793e+07,10.7212,-0.5,-0.500
498,2019-12-23,2019-12-23,2019-12-24,0.198899,2019-12-23,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,0.000121,-0.000571,0.002106,0.000121,FORCE_SELL,3.971139e+07,10.7325,-0.5,-0.500
499,2019-12-24,2019-12-24,2019-12-26,0.169106,2019-12-24,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,0.029743,0.006162,0.001980,0.029743,FORCE_BUY,4.089254e+07,11.0787,-0.5,1.655
500,2019-12-26,2019-12-26,2019-12-27,0.195538,2019-12-26,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,-0.014070,0.000797,-0.000117,-0.014070,FREE,4.092512e+07,10.9525,-0.5,1.655
501,2019-12-27,2019-12-27,2019-12-30,0.265744,2019-12-27,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_SELL,-0.021668,-0.023506,0.000118,0.000118,FORCE_SELL,4.092995e+07,10.7775,-0.5,1.655


,date,equity,cash,pos,qty,entry_px,decision_date,result_date,buy_th,sell_th,...,daily_buy_level,daily_sell_level,decision_extreme_base_dir,decision_extreme_region,decision_extreme_label,decision_extreme_window_end_date,decision_extreme_ref_high,decision_extreme_ref_low,decision_extreme_future_max_high,decision_extreme_future_min_low
497,2019-12-23,239222.816720,0.000000,1,22313.06353,10.51,2019-12-20,2019-12-23,-0.5,-0.500,...,0.200,0.250,sell,sell_high_broken,0.0,2019-12-30,10.6900,10.5750,11.1325,10.6
498,2019-12-24,239474.954338,0.000000,1,22313.06353,10.51,2019-12-23,2019-12-24,-0.5,1.655,...,0.315,0.335,sell,sell_high_broken,0.0,2019-12-31,10.7725,10.6962,11.1325,10.6
499,2019-12-26,247199.736932,0.000000,1,22313.06353,10.51,2019-12-24,2019-12-26,-0.5,1.655,...,0.315,0.335,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
500,2019-12-27,244383.828314,0.000000,1,22313.06353,10.51,2019-12-26,2019-12-27,-0.5,1.655,...,0.200,0.250,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
501,2019-12-30,245834.177444,245834.177444,0,0.00000,NaN,2019-12-27,2019-12-30,-0.5,2.070,...,0.200,0.250,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN


[TRAIN][DAILY-PROB] n=1861 pos=372 (19.99%)
[TRAIN][DAILY-PROB] n=1886 pos=377 (19.99%)
[TRAIN][DAILY-PROB] n=1911 pos=379 (19.83%)
[TRAIN][DAILY-PROB] n=1936 pos=380 (19.63%)
[TRAIN][DAILY-PROB] n=1961 pos=382 (19.48%)
[TRAIN][DAILY-PROB] n=1986 pos=386 (19.44%)
[TRAIN][DAILY-PROB] n=2011 pos=389 (19.34%)
[TRAIN][DAILY-PROB] n=2036 pos=395 (19.40%)
[TRAIN][DAILY-PROB] n=2061 pos=406 (19.70%)
[TRAIN][DAILY-PROB] n=2086 pos=407 (19.51%)
[TRAIN][DAILY-PROB] n=2111 pos=410 (19.42%)
[TRAIN][DAILY-PROB] n=2136 pos=418 (19.57%)
[TRAIN][DAILY-PROB] n=2161 pos=421 (19.48%)
[TRAIN][DAILY-PROB] n=2186 pos=427 (19.53%)
[TRAIN][DAILY-PROB] n=2211 pos=427 (19.31%)
[TRAIN][DAILY-PROB] n=2236 pos=430 (19.23%)
[TRAIN][DAILY-PROB] n=2261 pos=433 (19.15%)
[TRAIN][DAILY-PROB] n=2286 pos=444 (19.42%)
[TRAIN][DAILY-PROB] n=2311 pos=447 (19.34%)
[TRAIN][DAILY-PROB] n=2336 pos=453 (19.39%)
[TRAIN][DAILY-PROB] n=2361 pos=459 (19.44%)
[TRAIN][DAILY-PROB] n=2386 pos=461 (19.32%)
[TRAIN][DAILY-PROB] n=2411 pos=4

,date,decision_date,result_date,p_day,p_day_source_date,daily_gate_mode,daily_reward_mode,direct_gate_confidence,direct_gate_prob_force_buy,direct_gate_prob_free,...,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
2118,2026-06-05,2026-06-05,2026-06-08,0.380011,2026-06-05,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_SELL,0.017305,0.011502,0.011502,0.011502,FORCE_BUY,2.196337e+19,76.0639,-0.5,-0.5
2119,2026-06-08,2026-06-08,2026-06-09,0.278875,2026-06-08,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,-0.060625,0.005896,0.000000,-0.060625,FREE,2.209286e+19,73.2900,-0.5,-0.5
2120,2026-06-09,2026-06-09,2026-06-10,0.370303,2026-06-09,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_SELL,-0.060559,-0.063389,-0.016276,-0.016276,FORCE_SELL,2.173328e+19,68.1095,-0.5,-0.5
2121,2026-06-10,2026-06-10,2026-06-11,0.303717,2026-06-10,threshold,counterfactual_5m,NaN,NaN,NaN,...,FREE,0.070863,0.067446,0.000000,0.067446,FORCE_BUY,2.327336e+19,76.9629,-0.5,-0.5
2122,2026-06-11,2026-06-11,2026-06-12,0.283013,2026-06-11,threshold,counterfactual_5m,NaN,NaN,NaN,...,FREE,0.029686,0.072628,0.000000,0.072628,FREE,2.496366e+19,78.0399,-0.5,-0.5


,date,equity,cash,pos,qty,entry_px,decision_date,result_date,buy_th,sell_th,...,daily_buy_level,daily_sell_level,decision_extreme_base_dir,decision_extreme_region,decision_extreme_label,decision_extreme_window_end_date,decision_extreme_ref_high,decision_extreme_ref_low,decision_extreme_future_max_high,decision_extreme_future_min_low
1616,2026-06-08,2.356641e+06,2.356641e+06,0,0.000000,NaN,2026-06-05,2026-06-08,-0.5,-0.5,...,0.305,0.345,sell,sell_high_held,1.0,2026-06-12,82.08,72.68,79.3407,66.79
1617,2026-06-09,2.213769e+06,0.000000e+00,1,30205.607037,78.02,2026-06-08,2026-06-09,-0.5,-0.5,...,0.305,0.345,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1618,2026-06-10,2.154264e+06,2.154264e+06,0,0.000000,NaN,2026-06-09,2026-06-10,-0.5,-0.5,...,0.275,0.325,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1619,2026-06-11,2.226336e+06,2.226336e+06,0,0.000000,NaN,2026-06-10,2026-06-11,-0.5,-0.5,...,0.275,0.325,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1620,2026-06-12,2.388031e+06,2.388031e+06,0,0.000000,NaN,2026-06-11,2026-06-12,-0.5,-0.5,...,0.275,0.325,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx,exec_idx,exec_ts
0,buy,83316,11.0550,9045.680687,0.0,ADAPTIVE_FORCE_BUY->first acceptable 5m signal,2020-01-02 06:30:00,1.544917,-0.500,FORCE_BUY,NaN,NaN,NaN,83317,2020-01-02 06:35:00
1,sell,83669,10.8012,9045.680687,0.0,5m SELL signal,2020-01-06 05:00:00,3.807463,1.675,FREE,-2295.793758,11.0550,83316.0,83670,2020-01-06 05:05:00
2,buy,83677,10.8125,9036.227167,0.0,5m BUY signal,2020-01-06 05:45:00,3.590418,-0.500,FREE,NaN,NaN,NaN,83678,2020-01-06 05:55:00
3,sell,83687,10.8375,9036.227167,0.0,5m SELL signal,2020-01-06 07:05:00,3.758951,1.675,FREE,225.905679,10.8125,83677.0,83688,2020-01-06 07:10:00
4,buy,83715,10.7812,9083.414826,0.0,5m BUY signal,2020-01-06 09:25:00,3.157287,-0.500,FREE,NaN,NaN,NaN,83716,2020-01-06 09:30:00


In [ ]:

from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "TQQQ_adaptive_reward_fresh_start_252days_at_2020_W_Bonds.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/TQQQ_day.csv",
    k5m_csv_path="DataAPI/data/TQQQ_5M.csv",
    code="TQQQ",
    daily_chan_start="2012-01-01",
    accumulation_start="2018-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
        "US2Y": "US2Y.csv",
        #"US5Y": "US5Y.csv",
        "US10Y": "US10Y.csv",
        #"US30Y": "US30Y.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)

from adaptive_reward_checkpoint_fresh import run_adaptive_reward_from_snapshot

result = run_adaptive_reward_from_snapshot(
    **common_kwargs,
    snapshot_path="checkpoints/TQQQ_adaptive_reward_fresh_start_252days_at_2020__continued.joblib",
    end_time="2026-06-12",
    daily_gate_mode="free",    # no gate
    ret_model_type="lstm",     # "lstm" or "xgboost"
    lstm_seq_len=20,
    lstm_epochs=30,
    lstm_hidden_size=64,
    output_dir="output_5m_no_gate_lstm",
    verbose=True,
)